# Lab Assignment 7
## PCS221 Cloud Computing – MapReduce Programming

| Field | Details |
|---|---|
| **Name** | Writick Parui |
| **Roll No.** | 8025320111 |
| **Course** | PCS221 – Cloud Computing |

**Instructions:** Submit `.ipynb` file on Github  
**Reference Notebook:** [Google Colab](https://colab.research.google.com/drive/12UbpUk11eJwzQMzgMqey4yOkz_m0576c?authuser=2)

---
## Setup: Install mrjob

In [ ]:
!pip install mrjob -q
print("mrjob installed successfully.")

---
## Task 1: Basic MapReduce – Word Count

Implement a basic Word Count MapReduce job using `mrjob`.

**Concept:**
- **Mapper:** reads each line, emits `(word, 1)` for every word found
- **Reducer:** sums up counts for each word

We use a regex pattern `[\w']+` to extract words (handles contractions like `don't`).

In [ ]:
%%writefile wordcount.py
from mrjob.job import MRJob
import re

WORD_REGEXP = re.compile(r"[\w']+")

class MRWordCount(MRJob):

    def mapper(self, _, line):
        """Emit (word, 1) for each word in the line."""
        for word in WORD_REGEXP.findall(line):
            yield word.lower(), 1

    def reducer(self, word, counts):
        """Sum all counts for the word."""
        yield word, sum(counts)

if __name__ == '__main__':
    MRWordCount.run()

In [ ]:
# Quick demo on a small sample
with open('sample.txt', 'w') as f:
    f.write("hadoop is fast\nhadoop is scalable\nspark is also fast")

print("Task 1 – Word Count on sample input:")
!python wordcount.py sample.txt

---
## Task 2: Word Count on *Alice's Adventures in Wonderland*

Download the text from Project Gutenberg and run the Word Count MapReduce job.

**Question:** How many times does the word **Cheshire** occur?
> Note: The string `'Cheshire` (with a preceding apostrophe) does **not** count.

In [ ]:
# Download Alice's Adventures in Wonderland from Project Gutenberg
!wget -q -O alice.txt http://www.gutenberg.org/cache/epub/11/pg11.txt
print("Download complete!")

# Preview first 5 non-empty lines
print("\nFirst 5 lines of alice.txt:")
with open('alice.txt', 'r', encoding='utf-8') as f:
    count = 0
    for line in f:
        if line.strip():
            print(' ', line.strip())
            count += 1
            if count == 5:
                break

In [ ]:
# Run MapReduce Word Count on alice.txt
!python wordcount.py alice.txt > alice_wordcount_output.txt
print("MapReduce Word Count job complete!")

# Show total unique words found
with open('alice_wordcount_output.txt') as f:
    lines = f.readlines()
print(f"Total unique words: {len(lines)}")

In [ ]:
# Parse mrjob output to find count for 'cheshire'
# mrjob outputs tab-separated: "word"\tcount
cheshire_count = 0

with open('alice_wordcount_output.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split('\t')
            if len(parts) == 2:
                word = parts[0].strip('"')   # remove JSON quotes added by mrjob
                if word == 'cheshire':
                    cheshire_count = int(parts[1])

print(f"MapReduce result — 'cheshire' count: {cheshire_count}")

In [ ]:
# Verification using Python regex
# The mrjob wordcount regex [\w']+ treats apostrophes as word characters,
# so 'Cheshire is tokenized as a single word starting with '
# and is counted separately from Cheshire — which is exactly what we want.
import re

with open('alice.txt', 'r', encoding='utf-8') as f:
    text = f.read()

WORD_REGEXP = re.compile(r"[\w']+")
all_words   = WORD_REGEXP.findall(text)

# Count 'cheshire' (lowercase) — this excludes 'cheshire (starts with ')
cheshire_exact = sum(1 for w in all_words if w.lower() == 'cheshire')
apostrophe_ver = sum(1 for w in all_words if w.lower() == "'cheshire")

print(f"Verification:")
print(f"  Words matching 'cheshire'  (no apostrophe) : {cheshire_exact}")
print(f"  Words matching \'cheshire  (with apostrophe): {apostrophe_ver}")
print(f"\n>>> Answer: The word 'Cheshire' occurs {cheshire_exact} times.")

---
## Task 3: Word Median using MapReduce

The `wordmedian` application computes the **median length of words** in a text file.

**Input:** `words.txt` from [Google Drive](https://drive.google.com/file/d/16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas/view?usp=sharing)

**Question:** What is the median word length?

**Approach (2-step MapReduce):**
- Step 1 Mapper  → emit `(word_length, 1)` for each word
- Step 1 Reducer → emit `(None, (length, total_count))` per length bucket
- Step 2 Reducer → collect all buckets, sort by length, walk cumulative counts to find median position

In [ ]:
# Download words.txt from Google Drive
!pip install gdown -q
import gdown

file_id = '16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'words.txt', quiet=False)
print("\nwords.txt downloaded!")

# Preview
with open('words.txt', 'r') as f:
    lines = f.readlines()[:5]
print("\nFirst 5 lines:", lines)

In [ ]:
%%writefile wordmedian.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

WORD_REGEXP = re.compile(r"[\w']+")

class MRWordMedian(MRJob):
    """
    Two-step MapReduce job to compute the median word length.

    Step 1:
      Mapper  – emit (length, 1) for each word
      Reducer – aggregate counts per length bucket,
                emit (None, (length, total_count))

    Step 2:
      Reducer – collect all (length, count) pairs,
                sort by length, walk cumulative sum to find median
    """

    def steps(self):
        return [
            MRStep(mapper=self.mapper_word_length,
                   reducer=self.reducer_count_lengths),
            MRStep(reducer=self.reducer_find_median)
        ]

    # ── Step 1 ──────────────────────────────────────────────
    def mapper_word_length(self, _, line):
        for word in WORD_REGEXP.findall(line):
            yield len(word), 1

    def reducer_count_lengths(self, length, counts):
        # Pass (length, total) to a single reducer in Step 2
        yield None, (length, sum(counts))

    # ── Step 2 ──────────────────────────────────────────────
    def reducer_find_median(self, _, length_count_pairs):
        pairs = sorted(length_count_pairs, key=lambda x: x[0])

        total_words = sum(count for _, count in pairs)
        median_pos  = (total_words + 1) / 2.0   # 1-indexed median position

        cumulative = 0
        for length, count in pairs:
            cumulative += count
            if cumulative >= median_pos:
                yield 'median_word_length', length
                return

if __name__ == '__main__':
    MRWordMedian.run()

In [ ]:
# Run the wordmedian MapReduce job on words.txt
print("Running wordmedian MapReduce job...")
!python wordmedian.py words.txt

In [ ]:
# Verification using pure Python statistics library
import re
import statistics

WORD_REGEXP = re.compile(r"[\w']+")

with open('words.txt', 'r') as f:
    text = f.read()

word_lengths = [len(word) for word in WORD_REGEXP.findall(text)]

print(f"Total words          : {len(word_lengths)}")
print(f"Min word length      : {min(word_lengths)}")
print(f"Max word length      : {max(word_lengths)}")
print(f"Mean word length     : {statistics.mean(word_lengths):.4f}")
print(f"\n>>> Answer: Median word length = {statistics.median(word_lengths)}")

---
## Summary of Results

| Task | Description | Result |
|------|-------------|--------|
| Task 1 | Basic Word Count using mrjob | ✅ Implemented |
| Task 2 | Count of **'Cheshire'** in *Alice's Adventures in Wonderland* (no apostrophe) | *(run cells to see output)* |
| Task 3 | Median word length in `words.txt` | *(run cells to see output)* |

> **Note:** Run all cells top-to-bottom in Google Colab to populate results.